# Model PreTraining for Generation with HydraBLE (Augmented)

In [1]:
import os
from pathlib import Path


os.chdir(Path.cwd().parents[0])
print("Now in:", Path.cwd())

dataPathProcessed = str(Path.cwd()) + r"\data\csv" + r"\Processed Files\\"
checkPointPath = str(Path.cwd()) + r"\out\modeling\checkpoints\pretraining\\"

Now in: C:\Users\stsax\OneDrive\Studium\9. Semester\Masterarbeit\Repository


In [2]:
MAX_TOKEN_LENGTH = 1024
MAX_SEQUENCE_LENGTH = 32

In [3]:
import pickle
import pandas as pd
from modeling.BleDataset import BLEStreamDataset
import torch

from bpe.bpe import BleBytePairEncoder
from modeling.Trainer import TrainerConfig

MaskConfigPath = str(Path.cwd()) + r"\data_masking\mask_configs\\"
picklePath = str(Path.cwd()) + r"\out\pickle_objects\bpe" + "\\"

with open(picklePath + 'Fitted_BPE_State_Dict.pickle', 'rb') as f:
    state_dict = pickle.load(f)


pkt_df_train = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_train.parquet")
seq_table_train = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_v2_train.parquet")
stream_idx_train = pd.read_parquet(dataPathProcessed + r"classification_stream_index_v2_train.parquet")


pkt_df_val = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_validation.parquet")
seq_table_val = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_v2_validation.parquet")
stream_idx_val = pd.read_parquet(dataPathProcessed + r"classification_stream_index_v2_validation.parquet")


dataset_train = BLEStreamDataset(packet_df=pkt_df_train, sequence_table=seq_table_train, stream_index=stream_idx_train,
                           max_sequence_length=MAX_SEQUENCE_LENGTH, max_token_length=MAX_TOKEN_LENGTH,
                           tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type='train', augment_data=True, deterministic=False)

dataset_val = BLEStreamDataset(packet_df=pkt_df_val, sequence_table=seq_table_val, stream_index=stream_idx_val,
                           max_sequence_length=MAX_SEQUENCE_LENGTH, max_token_length=MAX_TOKEN_LENGTH,
                           tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type='validation', augment_data=True)

In [4]:
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=50, num_workers=4, pin_memory=True,     persistent_workers=True,     drop_last=False,     prefetch_factor=4, shuffle=True)

validation_loader = torch.utils.data.DataLoader(dataset_val, batch_size=50, num_workers=4, pin_memory=True,     persistent_workers=True,     drop_last=False,     prefetch_factor=4, shuffle=True)

In [5]:
from modeling.HydraBLE import HydraBLETransformer

bpe = BleBytePairEncoder.from_state_dict(state_dict)

model = HydraBLETransformer(
        vocab_size=bpe.vocab_size,
        num_classes=dataset_train.num_of_known_classes,
        pad_id=1,
        max_heads=8,
        head_dim=64,
        depth=4,
        max_len=2048,
        subnet_heads=[1, 2, 4, 8],
    )

ckpt = torch.load(str(Path.cwd()) + r"\out\modeling\checkpoints\pretraining\\"  + "last.pt", map_location="cuda:1")
model.load_state_dict(ckpt["model_state_dict"])


<All keys matched successfully>

In [6]:
from modeling.PreTrainingTrainer import PretrainingTrainerConfig, PretrainingTrainer

config = PretrainingTrainerConfig()

config.checkpoint_dir = checkPointPath
config.lr = 1e-2 * 5
config.epochs = 20
config.device =  "cuda:1"
config.subnet_heads = [1, 2, 4, 8]
config.max_ar_eval_batches = 0
config.ar_prefix_len = 30

trainer = PretrainingTrainer(model, train_loader, validation_loader, config)

In [ ]:
trainer.fit()

  8%|▊         | 49/600 [01:16<20:10,  2.20s/it] 

[train-pre] epoch=1 step=50 active_heads=8, loss=0.6754


 16%|█▋        | 99/600 [02:26<10:17,  1.23s/it]

[train-pre] epoch=1 step=100 active_heads=8, loss=0.6697


 25%|██▍       | 149/600 [03:40<13:01,  1.73s/it]

...

100%|██████████| 600/600 [14:45<00:00,  1.48s/it]


Start Validation


0it [00:07, ?it/s]



[epoch 2] train_loss=0.6680
  [val h=8] loss=0.0000 ar_token_acc=0.0000
  best_loss@h=8: 0.0000



  4%|▍         | 24/600 [00:39<08:49,  1.09it/s] 